# Audit de la prévision principale (§V-A, Tableau II de l'article)

Le Tableau II vient de `rf_forecast_v2.py` (`rf_forecast_results/results_v2_h3.json`) :
entraînement < 2021, calibration et seuil sur 2021–2022, test ≥ 2023 (7 632 jours-zones).
L'article annonce aussi « strict temporal cross-validation : AUC 0,861 ± 0,015 ».

Points audités :
1. **La « CV temporelle » de `rf_forecast_v2.py`** : `TimeSeriesSplit` est appliqué à des lignes
   triées **par zone puis par date** → les plis séparent des *zones*, pas des *périodes*.
2. **Baselines manquantes** : climatologie (saison) et persistance continue. L'importance des
   variables est dominée par le jour de l'année : que vaut le modèle face à la simple saison ?
3. **Définition de la cible** : seuils P97/P95 calculés sur toute l'année (saison sèche comprise).
4. Hétérogénéité par zone et intervalles de confiance du test.

Le test 2023–2025 et les modèles sont ceux de l'article (mêmes hyperparamètres retenus :
RF depth=10 leaf=10, HGB depth=5 lr=0,05, régression logistique).

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
ROOT = Path.cwd(); sys.path.insert(0, str(ROOT))
import rf_forecast as base
import rf_forecast_v2 as v2
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, average_precision_score
SEED = 42
OUT = ROOT / "rf_forecast_results" / "audit_prevision"; OUT.mkdir(parents=True, exist_ok=True)

rows, TH = v2.build_dataset(3)
train = rows[rows.date < v2.VAL_START]
val = rows[(rows.date >= v2.VAL_START) & (rows.date < v2.SPLIT_DATE)]
test = rows[rows.date >= v2.SPLIT_DATE]
F = v2.FEATURES
print(f"train {len(train)} | val {len(val)} | test {len(test)} (taux {test.flood_risk.mean():.3f})")

✓ Anti-fuite : aucune feature ne provient de la fenêtre-cible.
train 52704 | val 6588 | test 7632 (taux 0.194)


## 1. Reproduction du Tableau II

In [2]:
FACT = {"Random Forest": v2.rf_factory(10, 10), "HistGradientBoosting": v2.hgb_factory(5, 0.05),
        "Régression logistique": lambda: make_pipeline(StandardScaler(),
                                   LogisticRegression(max_iter=1000, class_weight="balanced"))}
Xtr, ytr = v2.matrix(train); Xval, yval = v2.matrix(val); Xte, yte = v2.matrix(test)
proba_te = {}
tab2 = []
for name, fac in FACT.items():
    est = fac().fit(Xtr, ytr)
    cal = CalibratedClassifierCV(est, method="isotonic", cv="prefit").fit(Xval, yval)
    p = cal.predict_proba(Xte)[:, 1]; proba_te[name] = p
    thr = base.pick_threshold_for_recall(yval, cal.predict_proba(Xval)[:, 1], 0.90)
    m = base.metrics_at_threshold(yte, p, thr)
    tab2.append(dict(Modèle=name, AUC=m["roc_auc"], PR_AUC=m["pr_auc"], Précision=m["precision"],
                     Rappel=m["recall"], F1=m["f1"]))
tab2 = pd.DataFrame(tab2).set_index("Modèle")
print("Article : RF 0,801 / HGB 0,804 / LogReg 0,810 (AUC)")
tab2.round(3)

Article : RF 0,801 / HGB 0,804 / LogReg 0,810 (AUC)


,AUC,PR_AUC,Précision,Rappel,F1
Modèle,,,,,
Random Forest,0.801,0.417,0.298,0.930,0.451
HistGradientBoosting,0.804,0.422,0.280,0.967,0.434
Régression logistique,0.810,0.440,0.323,0.884,0.474


## 2. La « validation croisée temporelle » de l'article

(a) protocole de `rf_forecast_v2.py` tel quel (lignes triées par zone) ;
(b) même `TimeSeriesSplit` mais lignes **triées par date** ;
(c) plis **par années** (entraînement sur les années passées, validation sur 2 années suivantes).

In [3]:
def cv(fac, df, splits):
    s = []
    for tr, va in splits:
        m = fac().fit(v2.matrix(df.iloc[tr])[0], df.iloc[tr].flood_risk)
        s.append(roc_auc_score(df.iloc[va].flood_risk, m.predict_proba(v2.matrix(df.iloc[va])[0])[:, 1]))
    return np.array(s)

tr_zone = train.reset_index(drop=True)                                   # ordre de l'article (zone, date)
tr_date = train.sort_values(["date", "location"]).reset_index(drop=True)
year_folds = []
for vy in [(2013, 2014), (2015, 2016), (2017, 2018), (2019, 2020)]:
    yrs = tr_date.date.dt.year
    year_folds.append((np.where(yrs < vy[0])[0], np.where(yrs.isin(vy))[0]))

res_cv = []
for name, fac in FACT.items():
    a = cv(fac, tr_zone, TimeSeriesSplit(4).split(tr_zone))
    b = cv(fac, tr_date, TimeSeriesSplit(4).split(tr_date))
    c = cv(fac, tr_date, year_folds)
    res_cv.append({"Modèle": name,
                   "(a) article : plis = zones": f"{a.mean():.3f} ± {a.std():.3f}",
                   "(b) TimeSeriesSplit par date": f"{b.mean():.3f} ± {b.std():.3f}",
                   "(c) plis par années": f"{c.mean():.3f} ± {c.std():.3f}"})
res_cv = pd.DataFrame(res_cv).set_index("Modèle")
res_cv

,(a) article : plis = zones,(b) TimeSeriesSplit par date,(c) plis par années
Modèle,,,
Random Forest,0.861 ± 0.015,0.840 ± 0.017,0.850 ± 0.015
HistGradientBoosting,0.862 ± 0.013,0.829 ± 0.026,0.846 ± 0.016
Régression logistique,0.819 ± 0.061,0.846 ± 0.015,0.852 ± 0.023


## 3. Baselines : climatologie et persistance continue (test 2023–2025)

- **Climatologie** : fréquence de la cible par zone et période de l'année (± 15 jours), apprise sur
  les années d'entraînement (< 2021). Aucune information météo du jour.
- **Persistance continue** : cumul des 3 derniers jours / P95 de la zone (l'article utilise une
  version binaire, d'où son AUC de 0,584).
- **Modèle sans saison** : Random Forest sans les variables jour-de-l'année (sin/cos, mois).

In [4]:
def climatology(fit_df, apply_df):
    fit_df = fit_df.assign(doy=fit_df.date.dt.dayofyear)
    cl = {}
    for z, g in fit_df.groupby("location"):
        s = g.groupby("doy").flood_risk.agg(["sum", "count"]).reindex(range(1, 367), fill_value=0)
        pad = lambda v: np.r_[v[-15:], v, v[:15]]
        num = np.convolve(pad(s["sum"].values), np.ones(31), "valid")
        den = np.convolve(pad(s["count"].values), np.ones(31), "valid")
        cl[z] = num / np.maximum(den, 1)
    return np.array([cl[z][d - 1] for z, d in zip(apply_df.location, apply_df.date.dt.dayofyear)])

P95 = {z: b for z, (a, b) in TH.items()}
scores_te = dict(proba_te)
scores_te["Climatologie"] = climatology(train, test)
scores_te["Persistance continue"] = (test.precip_3d_sum / test.location.map(P95)).to_numpy()
scores_te["Persistance binaire (article)"] = (test.precip_3d_sum >= test.location.map(P95)).astype(float).to_numpy()

NOSEAS = [f for f in F if f not in ("dayofyear_sin", "dayofyear_cos", "is_rainy_season", "month")]
m = v2.rf_factory(10, 10)().fit(train[NOSEAS].to_numpy(float), ytr)
scores_te["Random Forest sans variables de saison"] = m.predict_proba(test[NOSEAS].to_numpy(float))[:, 1]

# bootstrap par date
rng = np.random.default_rng(SEED)
groups = [np.where(test.date.values == d)[0] for d in np.unique(test.date.values)]
boot = {k: [] for k in scores_te}
for _ in range(1000):
    ix = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
    for k, s in scores_te.items():
        boot[k].append(roc_auc_score(yte[ix], s[ix]))
base_tab = pd.DataFrame([dict(Score=k, AUC=roc_auc_score(yte, s), IC95_bas=np.percentile(boot[k], 2.5),
                              IC95_haut=np.percentile(boot[k], 97.5), PR_AUC=average_precision_score(yte, s))
                         for k, s in scores_te.items()]).set_index("Score")
d = np.array(boot["Random Forest"]) - np.array(boot["Climatologie"])
print(f"RF − climatologie : ΔAUC = {d.mean():+.3f} [{np.percentile(d, 2.5):+.3f} ; {np.percentile(d, 97.5):+.3f}]")
d = np.array(boot["Régression logistique"]) - np.array(boot["Climatologie"])
print(f"LogReg − climatologie : ΔAUC = {d.mean():+.3f} [{np.percentile(d, 2.5):+.3f} ; {np.percentile(d, 97.5):+.3f}]")
base_tab.round(3)

RF − climatologie : ΔAUC = +0.009 [-0.002 ; +0.019]
LogReg − climatologie : ΔAUC = +0.017 [+0.005 ; +0.031]


,AUC,IC95_bas,IC95_haut,PR_AUC
Score,,,,
Random Forest,0.801,0.775,0.826,0.417
HistGradientBoosting,0.804,0.779,0.829,0.422
Régression logistique,0.810,0.783,0.835,0.440
Climatologie,0.793,0.765,0.818,0.394
Persistance continue,0.740,0.711,0.769,0.347
Persistance binaire (article),0.584,0.561,0.608,0.246
Random Forest sans variables de saison,0.795,0.771,0.818,0.419


**Lecture.** L'AUC mélange deux choses : savoir *quelle période de la saison* est pluvieuse
(la climatologie le sait déjà) et savoir *quels jours précis* le seront. Pour isoler la seconde,
on calcule l'AUC **à l'intérieur de chaque mois** (même saison, même zone mélangées).

In [5]:
def auc_within_month(y, s, month):
    out = []
    for mth in sorted(set(month)):
        k = month == mth
        if len(set(y[k])) == 2 and y[k].sum() >= 20:
            out.append((mth, roc_auc_score(y[k], s[k]), int(k.sum())))
    return out
month = test.date.dt.month.to_numpy()
wm = {}
for k in ["Random Forest", "Régression logistique", "Climatologie", "Persistance continue"]:
    r = auc_within_month(yte, scores_te[k], month)
    wm[k] = {f"mois {mth}": a for mth, a, n in r}
    wm[k]["moyenne pondérée"] = np.average([a for _, a, _ in r], weights=[n for *_, n in r])
pd.DataFrame(wm).T.round(3)

,mois 6,mois 7,mois 8,mois 9,mois 10,moyenne pondérée
Random Forest,0.655,0.618,0.617,0.628,0.730,0.648
Régression logistique,0.706,0.629,0.673,0.629,0.771,0.679
Climatologie,0.724,0.614,0.544,0.605,0.703,0.643
Persistance continue,0.631,0.545,0.576,0.566,0.729,0.606


## 4. Définition de la cible : seuils calculés sur toute l'année

In [6]:
thr = pd.DataFrame(TH, index=["P97 pluie/jour (mm)", "P95 cumul 3 j (mm)"]).T
thr["taux de cible (saison, test)"] = test.groupby("location").flood_risk.mean()
# seuils alternatifs : saison des pluies uniquement
cal_rs = base.calibrate_thresholds(base.add_features(base.aggregate_daily(base.load_hourly(base.DATA_DIR)))
                                   .pipe(lambda d: d[(d.date < v2.SPLIT_DATE) & d.month.isin(base.RAINY_MONTHS)]), 3)
thr["P97 saison des pluies (mm)"] = pd.Series({z: a for z, (a, b) in cal_rs.items()})
thr["P95 3 j saison des pluies (mm)"] = pd.Series({z: b for z, (a, b) in cal_rs.items()})
thr.sort_values("P97 pluie/jour (mm)").round(1)

,P97 pluie/jour (mm),P95 cumul 3 j (mm),"taux de cible (saison, test)",P97 saison des pluies (mm),P95 3 j saison des pluies (mm)
podor,3.0,5.9,0.2,7.0,13.2
saint_louis,4.7,9.5,0.1,9.2,15.7
matam,6.1,11.8,0.3,11.3,20.7
louga,7.8,13.9,0.1,12.4,21.5
dakar_banlieue,8.9,16.7,0.1,13.6,26.4
bakel,10.5,18.9,0.2,15.9,29.9
thies,10.8,21.3,0.1,16.0,31.3
diourbel,11.5,22.5,0.2,16.4,30.8
mbacke,11.8,21.7,0.2,16.3,31.1
touba,12.0,21.5,0.2,16.2,30.8


In [7]:
# Même expérience (RF de l'article) avec la cible recalée sur la saison des pluies
daily_rs = base.add_future_target(base.add_features(base.aggregate_daily(base.load_hourly(base.DATA_DIR))), 3, cal_rs)
rows_rs = daily_rs[daily_rs.month.isin(base.RAINY_MONTHS)].dropna(subset=["flood_risk"]).copy()
rows_rs["flood_risk"] = rows_rs.flood_risk.astype(int)
tr_rs, te_rs = rows_rs[rows_rs.date < v2.VAL_START], rows_rs[rows_rs.date >= v2.SPLIT_DATE]
m = v2.rf_factory(10, 10)().fit(v2.matrix(tr_rs)[0], tr_rs.flood_risk)
p = m.predict_proba(v2.matrix(te_rs)[0])[:, 1]
print(f"Cible 'saison des pluies' : taux test = {te_rs.flood_risk.mean():.3f} (contre {test.flood_risk.mean():.3f})")
print(f"  RF      AUC = {roc_auc_score(te_rs.flood_risk, p):.3f}  PR-AUC = {average_precision_score(te_rs.flood_risk, p):.3f}")
cl = climatology(tr_rs, te_rs)
print(f"  Climato AUC = {roc_auc_score(te_rs.flood_risk, cl):.3f}")

Cible 'saison des pluies' : taux test = 0.117 (contre 0.194)
  RF      AUC = 0.792  PR-AUC = 0.269


  Climato AUC = 0.782


## 5. Hétérogénéité par zone (test 2023–2025, Random Forest)

In [8]:
pz = pd.DataFrame({"zone": test.location.values, "y": yte, "rf": proba_te["Random Forest"],
                   "clim": scores_te["Climatologie"]})
per_zone = pz.groupby("zone").apply(lambda g: pd.Series({
    "taux": g.y.mean(), "AUC RF": roc_auc_score(g.y, g.rf) if g.y.nunique() == 2 else np.nan,
    "AUC climato": roc_auc_score(g.y, g.clim) if g.y.nunique() == 2 else np.nan}))
per_zone.sort_values("AUC RF").round(3)

,taux,AUC RF,AUC climato
zone,,,
touba,0.170,0.755,0.737
sedhiou,0.224,0.759,0.767
ziguinchor,0.215,0.768,0.762
kolda,0.259,0.770,0.776
kaolack_leona,0.217,0.775,0.761
matam,0.252,0.776,0.778
bakel,0.231,0.783,0.769
tambacounda,0.262,0.786,0.794
louga,0.113,0.798,0.778


In [9]:
tab2.to_csv(OUT / "tableau2_reproduit.csv"); res_cv.to_csv(OUT / "cv_protocoles.csv")
base_tab.to_csv(OUT / "baselines_test.csv"); thr.to_csv(OUT / "seuils_cible.csv"); per_zone.to_csv(OUT / "auc_par_zone.csv")
print("✓", OUT)

✓ /Users/mac/Documents/Innond/flood_api/rf_forecast_results/audit_prevision


## 6. Conclusions (exécution du 25/09/2026)

**1. Tableau II reproduit à l'identique** (RF 0,801 / HGB 0,804 / LogReg 0,810). Le test
2023–2025 est bien un holdout temporel : ces chiffres sont valides.

**2. La « strict temporal cross-validation » (0,861 ± 0,015) est mal nommée.** Les plis de
`rf_forecast_v2.py` séparent des **zones** sur les mêmes années (lignes triées par zone). Une vraie
CV temporelle donne **0,850 ± 0,015** (RF, plis par années) ou 0,840 (TimeSeriesSplit par date) :
l'ordre de grandeur reste, mais il faut corriger le libellé et la valeur. La régression
logistique passe de 0,819 à 0,852 : elle n'est pas inférieure aux ensembles.

**3. Constat majeur : sans la prévision météo, le modèle fait à peine mieux que la climatologie.**
Sur le test 2023–2025, la simple fréquence saisonnière par zone donne **AUC 0,793** contre 0,801
pour le RF (Δ = +0,009 [−0,002 ; +0,019], non significatif) et 0,810 pour la régression logistique
(+0,017, significatif). À l'intérieur d'un même mois, l'AUC tombe à ~0,65–0,68. Autrement dit,
l'essentiel des 0,80 vient de la **saisonnalité** (cohérent avec l'importance dominante du jour de
l'année). La persistance de l'article (0,584) est une baseline trop faible car binarisée ; en
continu elle vaut 0,740. → L'article doit ajouter la climatologie comme baseline et présenter
honnêtement que la compétence « jour par jour » vient surtout de la **prévision numérique**
(cf. `validation_temporelle_prevision.ipynb` : +0,06 d'AUC, 0,852 contre 0,764 pour la
climatologie sur 2024–2026).

**4. Seuils de la cible.** Calculés sur toute l'année, ils sont très bas en zone aride (Podor :
3,0 mm/j) ; calculés sur la saison des pluies, ils doublent environ (Podor : 7,0 mm/j) et le taux
de cible passe de 19,4 % à 11,7 %. Les conclusions (RF ≈ climatologie, AUC ~0,79) ne changent pas :
c'est une limite de définition à signaler, pas une erreur qui invalide les résultats.

**5. Par zone** : AUC RF de 0,76 (Touba) à 0,86 (Podor), toujours proche de la climatologie
de la zone.